# Module 8 : Le Boss Final — Le secret mathématique 🤯

Avez-vous essayé de lancer votre BFS sur un jeu à 8 ou 9 disques ?
Votre ordinateur rame. C'est **l'explosion combinatoire** : pour n disques, il y a 3^n états possibles.

Pour 20 disques → 3 486 784 401 états. Impossible à explorer !

Il existe pourtant une formule mathématique qui calcule le coût minimal entre **n'importe quelle**
configuration de départ et d'arrivée, **sans rien explorer**. C'est le secret de ce module.

## 1. Mesurer l'explosion combinatoire

Avant de découvrir la formule magique, visualisons le problème.
Utilisons `time` pour mesurer le temps d'exécution du BFS selon n.

In [ ]:
import time
import matplotlib.pyplot as plt
from collections import deque

# Réimportation des fonctions utiles des modules précédents
def etat_initial(n): return (tuple(range(n, 0, -1)), (), ())
def etat_final(n):   return ((), (), tuple(range(n, 0, -1)))

def get_voisins(etat):
    voisins = []
    for i, j in [(0,1),(0,2),(1,0),(1,2),(2,0),(2,1)]:
        source, dest = etat[i], etat[j]
        if not source: continue
        disque = source[-1]
        if dest and disque > dest[-1]: continue
        e = [list(p) for p in etat]
        e[i].pop(); e[j].append(disque)
        voisins.append(tuple(tuple(p) for p in e))
    return voisins

def bfs_hanoi(etat_init, etat_fin):
    file = deque([(etat_init, 0)])
    visites = {etat_init: 0}
    while file:
        courant, dist = file.popleft()
        if courant == etat_fin: return dist
        for v in get_voisins(courant):
            if v not in visites:
                visites[v] = dist + 1
                file.append((v, dist + 1))
    return -1

valeurs_n = [2, 3, 4, 5, 6]
temps = []

for n in valeurs_n:
    debut = time.time()
    bfs_hanoi(etat_initial(n), etat_final(n))
    fin = time.time()
    temps.append(fin - debut)
    print(f"n={n} : {fin-debut:.4f}s  (3^n = {3**n} états)")

plt.figure(figsize=(8, 4))
plt.plot(valeurs_n, temps, 'r-o')
plt.xlabel("Nombre de disques"); plt.ylabel("Temps (s)")
plt.title("Temps BFS selon n — l'explosion combinatoire")
plt.grid(True, alpha=0.3); plt.show()

## 2. Le secret : localiser le plus gros disque

La clé de la formule mathématique est de regarder où se trouve le **plus gros disque** (disque n).

**Observation :** Peu importe la configuration des n-1 petits disques, le coût pour déplacer
le disque n de son piquet actuel vers sa destination passe toujours par une étape intermédiaire fixe.

## 🧩 Exercice 1 : Localiser le plus gros disque (★☆☆)

Créez une fonction `trouver_piquet(etat, disque)` qui retourne l'**index du piquet** (0, 1 ou 2)
sur lequel se trouve un disque donné.

In [ ]:
def trouver_piquet(etat, disque):
    for i, piquet in enumerate(etat):
        if disque in piquet:
            return i
    return -1   # Ne devrait jamais arriver

# Test :
etat_test = ((3, 2), (1,), ())
print(trouver_piquet(etat_test, 3))   # → 0 (sur A)
print(trouver_piquet(etat_test, 1))   # → 1 (sur B)

In [ ]:
etat_test = ((3, 2), (1,), ())
assert trouver_piquet(etat_test, 3) == 0
assert trouver_piquet(etat_test, 2) == 0
assert trouver_piquet(etat_test, 1) == 1
print("✅ Exercice 1 validé !")

## 3. La formule récursive

Voici l'idée centrale. Pour calculer le coût de transition de `etat_dep` vers `etat_arr` avec n disques :

**Cas 1 :** Le disque n est déjà au bon piquet → le coût est celui de transition des n-1 petits disques.

**Cas 2 :** Le disque n doit bouger. Dans ce cas :
- Les n-1 petits disques doivent d'abord être rassemblés sur le "piquet libre" (ni source ni destination du disque n).
- On déplace le disque n (1 coup).
- Les n-1 petits disques vont ensuite vers leur destination finale.

Le coût de "rassembler n-1 disques sur un piquet précis depuis n'importe quelle config" est `2^(n-1) - 1` au maximum.
Donc le coût total quand le disque n doit bouger = `(2^(n-1) - 1) + 1 + (2^(n-1) - 1)` = **`2^n - 1`**.

## 🧩 Exercice 2 : Implémenter la formule (★★★)

Créez la fonction `cout_minimal(etat_dep, etat_arr, n)` qui calcule le nombre minimal de coups
pour passer de `etat_dep` à `etat_arr` en utilisant la formule récursive.

**Cas de base :** n == 0 → coût = 0

**Sinon :**
- Trouver où est le disque n dans `etat_dep` (piquet source)
- Trouver où doit aller le disque n dans `etat_arr` (piquet destination)
- Si source == destination → coût = `cout_minimal(etat_dep, etat_arr, n-1)`
- Sinon → coût = `2^n - 1`

In [ ]:
def cout_minimal(etat_dep, etat_arr, n):
    if n == 0:
        return 0
    
    src = trouver_piquet(etat_dep, n)   # Où est le disque n au départ
    dst = trouver_piquet(etat_arr, n)   # Où doit aller le disque n
    
    if src == dst:
        # Le disque n est déjà en place, on résout récursivement pour n-1
        return cout_minimal(etat_dep, etat_arr, n-1)
    else:
        # Le disque n doit bouger : coût = 2^n - 1
        return 2**n - 1

# Test rapide :
dep = etat_initial(3)
arr = etat_final(3)
print(f"Coût minimal (formule) : {cout_minimal(dep, arr, 3)}")   # → 7
print(f"Coût BFS               : {bfs_hanoi(dep, arr)}")          # → 7

In [ ]:
assert cout_minimal(etat_initial(3), etat_final(3), 3) == 7
assert cout_minimal(etat_initial(1), etat_final(1), 1) == 1
assert cout_minimal(etat_initial(5), etat_final(5), 5) == 31
print("✅ Exercice 2 validé ! La formule donne les bons résultats.")

## 🧩 Exercice 3 : Vérification sur des configurations aléatoires (★★★★)

Pour vraiment valider la formule, testez-la sur des **configurations aléatoires** :
1. Générez 10 états de départ aléatoires valides pour n=4 disques
2. Pour chacun, comparez `cout_minimal(dep, etat_final(4), 4)` avec `bfs_hanoi(dep, etat_final(4))`
3. Les deux doivent toujours être égaux !

In [ ]:
import random

def generer_etat_aleatoire(n):
    """Génère une configuration aléatoire valide pour n disques."""
    piquets = [[], [], []]
    for disque in range(n, 0, -1):
        piquet_choisi = random.randint(0, 2)
        piquets[piquet_choisi].append(disque)
    # Trier chaque piquet (grand en bas, petit en haut) — garantit la validité
    for p in piquets:
        p.sort(reverse=True)
    return tuple(tuple(p) for p in piquets)

n = 4
arrivee = etat_final(n)
tous_corrects = True

print(f"Test sur 10 configurations aléatoires (n={n}) :")
for i in range(10):
    dep = generer_etat_aleatoire(n)
    formule = cout_minimal(dep, arrivee, n)
    bfs    = bfs_hanoi(dep, arrivee)
    ok = "✅" if formule == bfs else "❌"
    print(f"  Test {i+1} : formule={formule}, BFS={bfs} {ok}")
    if formule != bfs:
        tous_corrects = False

if tous_corrects:
    print("\n🎉 La formule est correcte sur tous les tests !")
else:
    print("\n⚠️ Certains tests ont échoué. Vérifiez votre implémentation.")

## 🧩 Exercice 4 : Comparaison finale des performances (★★★★)

La vraie puissance de la formule : comparez le **temps d'exécution** pour n=10, 15, 20 disques.
Le BFS est impossible (trop long), mais la formule répond instantanément !

In [ ]:
import time

print("Comparaison des performances :\n")
print(f"{'n':>4} | {'Formule (s)':>12} | {'3^n états':>12} | BFS")
print("-" * 50)

for n in [5, 8, 10, 15, 20]:
    dep = etat_initial(n)
    arr = etat_final(n)
    
    debut = time.time()
    resultat_formule = cout_minimal(dep, arr, n)
    temps_formule = time.time() - debut
    
    if n <= 6:
        debut = time.time()
        resultat_bfs = bfs_hanoi(dep, arr)
        temps_bfs = f"{time.time()-debut:.4f}s"
        egal = "✅" if resultat_formule == resultat_bfs else "❌"
    else:
        temps_bfs = "trop long"
        egal = ""
    
    print(f"{n:>4} | {temps_formule:>10.6f}s | {3**n:>12,} | {temps_bfs} {egal}")

## 🧩 Exercice Final : Le graphe de Hanoï est un triangle de Sierpiński (★★★★)

Voici le secret ultime des Tours de Hanoï : si l’on dessine le **graphe de tous les états possibles** (chaque état = un nœud, chaque coup légal = une arête), la forme obtenue est un **triangle de Sierpiński** — une fractale célèbre !

Pour 2 disques (9 états), le graphe ressemble à un triangle. Pour 3 disques (27 états), c’est un triangle fait de 3 triangles plus petits. Et ainsi de suite.

Nous allons vérifier cela visuellement avec `networkx` et `matplotlib`.

In [ ]:
# Si networkx n’est pas installé : pip install networkx
try:
    import networkx as nx
    NX_OK = True
except ImportError:
    NX_OK = False
    print("networkx non installé. Exécutez : pip install networkx")

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import deque

def get_voisins(etat):
    voisins = []
    for i, j in [(0,1),(0,2),(1,0),(1,2),(2,0),(2,1)]:
        source, dest = etat[i], etat[j]
        if not source: continue
        disque = source[-1]
        if dest and disque > dest[-1]: continue
        e = [list(p) for p in etat]
        e[i].pop(); e[j].append(disque)
        voisins.append(tuple(tuple(p) for p in e))
    return voisins

def construire_graphe_hanoi(n):
    """Construit le graphe complet de tous les états pour n disques."""
    G = nx.Graph()
    etat_init = (tuple(range(n, 0, -1)), (), ())
    file = deque([etat_init])
    visites = {etat_init}
    while file:
        courant = file.popleft()
        G.add_node(courant)
        for voisin in get_voisins(courant):
            G.add_edge(courant, voisin)
            if voisin not in visites:
                visites.add(voisin)
                file.append(voisin)
    print(f"Graphe Hanoï n={n} : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes (attendu : 3^{n}={3**n} états)")
    return G

if NX_OK:
    G2 = construire_graphe_hanoi(2)
    G3 = construire_graphe_hanoi(3)

In [ ]:
if NX_OK:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, G, n in [(axes[0], G2, 2), (axes[1], G3, 3)]:
        pos = nx.kamada_kawai_layout(G)
        couleurs = []
        for noeud in G.nodes():
            for idx, piquet in enumerate(noeud):
                if n in piquet:
                    couleurs.append(['#e74c3c', '#3498db', '#2ecc71'][idx])
                    break
        nx.draw(G, pos=pos, ax=ax,
                node_color=couleurs,
                node_size=80 if n == 3 else 200,
                edge_color='#bdc3c7',
                width=0.8,
                with_labels=False)
        ax.set_title(f"Graphe Hanoï n={n}  ({3**n} états) — Rouge=A, Bleu=B, Vert=C",
                     fontsize=10)

    plt.suptitle("Le graphe des Tours de Hanoï = Triangle de Sierpinski",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("Observation : les 3 grandes zones colorées correspondent aux 3 coins du triangle.")
    print("Chaque zone est elle-même un triangle plus petit : c'est la récursivité visible !")
else:
    print("Installez networkx pour voir la visualisation : pip install networkx")

## 🏆 Félicitations — Vous avez terminé le parcours !

Vous êtes parti de zéro et vous avez maintenant :

1. **Bases Python** — variables, boucles, fonctions
2. **POO** — classes, héritage, encapsulation
3. **Structures de données** — Pile (LIFO), File (FIFO)
4. **Récursivité** — un problème qui se résout lui-même
5. **Algorithmes de graphes** — BFS pour trouver le chemin le plus court
6. **Mathématiques algorithmiques** — une formule en O(n) qui bat un algorithme en O(3^n)

Les Tours de Hanoï ne sont pas qu'un jeu : elles cachent toute la richesse de l'informatique théorique.